# Code was run on Colab Pro

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import collections
import torch.optim as optim
from torch.optim import Optimizer
import time
import matplotlib.pyplot as plt

from AdamW          import AdamW
from utils          import utility, misreportUtility, misreportOptimization, trueUtility, loss
from networks       import AdditiveMechanism, Misreports,AllocationNet,PaymentNet
from restrictedAdam import Adam 

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.cuda.set_device(2)

# Set Random Seed 

In [ ]:
# Initializing seeds
torch.manual_seed(2)
np.random.seed(2)

# Testing Function

In [ ]:
def test(nBatch, nbrInit, R, gamma=0.001, minimum=0, maximum=1):
    
    """ This function computes the regret and payment of mechanism on a test set of size nBatch
        The optimal misreport is computed by optimizing the utility function (not by using the Misreport network)
        for R gradient steps (of stepsize gamma) and starting from nbrInit initialization, we only keep the best misreport
        To compute the regret we evaluate the mechanism at the misreport and compare to the valuation
        minimum and maximum indicate the range of the valuations
    """
    
    true = np.random.rand(nBatch,nAgent,nObject)

    localMisreports     = np.random.rand(nBatch,nbrInit,nAgent,nObject)
    batchMisreports     = torch.tensor(localMisreports).float().to(device)
    batchTrueValuations = torch.tensor(true).float().to(device)
    batchMisreports.requires_grad = True
    
    opt = Adam([batchMisreports], lr=gamma)
    
    for k in range(R):
        advU         = misreportUtility(mechanism,batchTrueValuations,batchMisreports)
        los          =  -1*torch.mean(advU).to(device)
        los.backward()
        opt.step(restricted= True, min=minimum, max=maximum)
        opt.zero_grad()
    
    misReportUtilityMax  = torch.max(advU, dim =1)[0]
    mechanism.zero_grad()
    allocation, payment = mechanism(batchTrueValuations)
    regret = F.relu(misReportUtilityMax -utility(batchTrueValuations, allocation, payment))
    mregret= torch.sum(torch.mean(regret, dim=0)).to(device)
    mregret= float(mregret.cpu().detach().numpy())

    with torch.no_grad():
        l,rMean,p = loss(payment, regret)

    testRegret.append(mregret)
    testPayment.append(float(p.detach().cpu().numpy() ))
    testOptimal.append(float((-l).detach().cpu().numpy())**2)
    print("Total regret: ",'{0:.5f}'.format(mregret), "Average regret per bidder: ",'{0:.5f}'.format(mregret/nAgent), " Optimal Revenue: ",'{0:.3f}'.format(float((-l).detach().cpu().numpy())**2), " payment: ",'{0:.3f}'.format(float(p.detach().cpu().numpy() )))

# Initializing Networks

In [ ]:
nAgent   = 3
nObject  = 10

# Parameters for the mechanism (payment and allocation network)
nLayersAllocation   = 7
nLayersPayment      = 7
widthAllocation     = 100
widthPayment        = 100

# Parameters for the misreport network
nLayersMisreport    = 7
widthMisreport      = 100

gamma              = 0.001 
testBatch          = 10000

nExperiments       = 200000
batchSize          = 500
nbrBatches         = int(nExperiments/batchSize)



alloc_net = AllocationNet(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
opt_alloc = AdamW(alloc_net.parameters(), lr=1e-3)

pay_net   = PaymentNet(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
opt_pay   = AdamW(pay_net.parameters(),   lr=1e-3)

mechanism            = AdditiveMechanism(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
optimizerMechanism   = AdamW(mechanism.parameters(), lr=0.001)

misreport            = Misreports(nAgent,nObject,nLayersMisreport, widthMisreport).to(device)
optimizerMisreport   = AdamW(misreport.parameters(), lr=0.001)

In [ ]:
testRegret    = []
testMaxRegret = []
testPayment   = []
testOptimal   = []
testTime      = []
testIteration = [0]

# range of valuations
minimum            = 0
maximum            = 1

In [ ]:
@torch.no_grad()
def myerson_itemwise_allocation_payment(values, reserve=0.5):
    """
    values:  (B, A, O)  估值矩阵
    reserve: 保留价
    return:  alloc (B, A, O), pay_myr (B, A)
    """
    B, A, O = values.shape
    # 逐物品取 top-2 出价
    top2 = values.topk(k=2, dim=1)
    v1, idx1 = top2.values[:, 0, :], top2.indices[:, 0, :]  # 最高价及其索引
    v2 = top2.values[:, 1, :]                               # 第二高价

    # 判断是否超过保留价
    win = (v1 >= reserve).float()        # (B,O)
    price = torch.maximum(v2, torch.full_like(v2, reserve)) * win

    # 构造 one-hot 分配矩阵
    alloc = torch.zeros(B, A, O, device=values.device)
    alloc.scatter_(1, idx1.unsqueeze(1), win.unsqueeze(1))

    # 计算每个代理的总支付
    pay_myr = alloc * price.unsqueeze(1)  # (B,A)
    return alloc, pay_myr

# Training

In [ ]:
R=10000
reserve=0.5
print("Train AllocationNet with Myerson supervision")
for t in range(1,60*nbrBatches+1):
    # 随机估值
    values = torch.rand(batchSize, nAgent, nObject, device=device)

    # 计算 Myerson 标签
    with torch.no_grad():
        alloc_myr, pay_myr = myerson_itemwise_allocation_payment(values, reserve=reserve)

    # 网络输出
    alloc_pred = alloc_net(values)

    # 监督损失
    loss_alloc = F.mse_loss(alloc_pred, alloc_myr)

    opt_alloc.zero_grad()
    loss_alloc.backward()
    opt_alloc.step()
    if t % (2*nbrBatches)==0 :
        print(f"loss={loss_alloc.item():.6f}")

In [ ]:
print("Train PaymentNet with Myerson supervision")

for t in range(1,60*nbrBatches+1):
    values = torch.rand(batchSize, nAgent, nObject, device=device)

    with torch.no_grad():
        alloc_myr, pay_myr = myerson_itemwise_allocation_payment(values, reserve=reserve)

    payments_pred = pay_net(values, alloc_myr)

    loss_pay = F.mse_loss(payments_pred, pay_myr)

    opt_pay.zero_grad()
    loss_pay.backward()
    opt_pay.step()
    if t % (2*nbrBatches)==0 :
        print(f"loss={loss_pay.item():.6f}")

In [ ]:
duration   = 0
R          = 100

i=0
mechanism            = AdditiveMechanism(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)

mechanism.alloc_net.load_state_dict(alloc_net.state_dict())
mechanism.payment_net.load_state_dict(pay_net.state_dict())
optimizerMechanism   = AdamW(mechanism.parameters(), lr=0.001)

print("Initial Test")
test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

for t in range(1,60*nbrBatches+1):
    
    # Reinitialize Misreport network periodically at the beginning of training
    if (t%(2*nbrBatches) ==1):
      if   t< 20*nbrBatches+2 :
    
        misreport            = Misreports(nAgent,nObject,nLayersMisreport, widthMisreport).to(device)
        optimizerMisreport   = AdamW(misreport.parameters(), lr=0.001)

    batchTrueValuations = torch.tensor(np.random.rand(batchSize,nAgent,nObject)).float().to(device)
    
    # Optimize Misreport Network for R steps
    for k in range(R):
  
        misreports          = misreport(batchTrueValuations).unsqueeze(1)
        mUtility            = misreportUtility(mechanism,batchTrueValuations,misreports).squeeze(1)
        mLoss               = torch.sum(torch.mean(-mUtility,dim=0))

        optimizerMisreport.zero_grad()
        mLoss.backward()
        optimizerMisreport.step()

    
    # Optimize Mechanism network for one step
    misreports          = misreport(batchTrueValuations).unsqueeze(1)
    mUtility            = misreportUtility(mechanism,batchTrueValuations,misreports).squeeze(1)

    allocation, payment = mechanism(batchTrueValuations)

    regret     = F.relu(mUtility -utility(batchTrueValuations, allocation, payment))
    l,rMean,p = loss(payment, regret)
        
    optimizerMechanism.zero_grad()

    l.backward()

    optimizerMechanism.step()
    
    # Test mechanism periodically
    if t % (2*nbrBatches)==0 :
        print("Batch: ", 2*int(t/(2*nbrBatches)))
        testTime.append(duration)
        testIteration.append(t/nbrBatches)
        test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

# Testing

In [ ]:
for i in range(200):
    test(50, nbrInit=1000, R=2000, gamma=0.001, minimum=0, maximum=1)

In [ ]:
totalregret = np.mean(np.array(testRegret[-200:]))
revenue     = np.mean(np.array(testPayment[-200:]))
print("Final Result")
print("Total Regret = ", '{0:.5f}'.format(totalregret), "Average regret per bidder: ",'{0:.5f}'.format(totalregret/nAgent), " Optimal Revenue: ",'{0:.3f}'.format(float(np.sqrt(revenue)-np.sqrt(totalregret))**2), " payment: ",'{0:.3f}'.format(revenue))

In [ ]:
stdregret = np.std(np.array(testRegret[-200:]))
stdrevenue= np.std(np.array(testPayment[-200:]))
print("std Regret = ", '{0:.5f}'.format(stdregret), "std regret per bidder: ",'{0:.5f}'.format(stdregret/nAgent), " std payment: ",'{0:.3f}'.format(stdrevenue))

In [ ]:
torch.save(mechanism,"3*10-7-100-2.pt")